<a href="https://colab.research.google.com/github/Surajsurya95096/My-Bot-Deployer/blob/main/deploy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# @title 🚀 **Universal Heroku Deployer (Public & Private Repo)** { display-mode: "form" }
# @markdown ### ⚙️ Heroku Details
Heroku_Email = ""  # @param {type:"string"}
Heroku_API_Key = ""  # @param {type:"string"}
Heroku_App_Name = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### 🔒 GitHub Repository (Private & Public)
Git_Repo_URL = ""  # @param {type:"string"}
Git_Branch = "master"  # @param {type:"string"}
GitHub_Personal_Access_Token = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### ⚙️ Dyno & Logs
Dyno_Type = "web"  # @param ["worker", "web"]
Auto_Scale = True  # @param {type:"boolean"}
Stream_Logs = True  # @param {type:"boolean"}

import os
import subprocess
import sys
import time

GREEN = "\033[92m"
RED = "\033[91m"
YELLOW = "\033[93m"
RESET = "\033[0m"


def run_cmd(cmd, check=True):
  res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
  if check and res.returncode != 0:
    print(f"{RED}Error: {res.stderr.strip()}{RESET}")
  return res.returncode, res.stdout, res.stderr


if (
    not Heroku_Email
    or not Heroku_API_Key
    or not Heroku_App_Name
    or not Git_Repo_URL
):
  print(
      f"{RED}[!] Saare required fields (Email, API Key, App Name, Repo URL) bharein!{RESET}"
  )
  sys.exit(1)

formatted_repo_url = Git_Repo_URL.strip()
if GitHub_Personal_Access_Token.strip():
  clean_url = (
      formatted_repo_url.replace("https://", "")
      .replace("http://", "")
      .split("@")[-1]
  )
  formatted_repo_url = (
      f"https://{GitHub_Personal_Access_Token.strip()}@{clean_url}"
  )

print(f"{YELLOW}[1/5] Heroku CLI verify ho rahi hai...{RESET}")
subprocess.run(
    "curl -s https://cli-assets.heroku.com/install.sh | sh > /dev/null 2>&1",
    shell=True,
)

print(f"{YELLOW}[2/5] Heroku credentials configure ho rahe hain...{RESET}")
netrc_data = f"machine api.heroku.com\n  login {Heroku_Email.strip()}\n  password {Heroku_API_Key.strip()}\nmachine git.heroku.com\n  login {Heroku_Email.strip()}\n  password {Heroku_API_Key.strip()}\n"
with open(os.path.expanduser("~/.netrc"), "w") as f:
  f.write(netrc_data)
os.chmod(os.path.expanduser("~/.netrc"), 0o600)

print(f"{YELLOW}[3/5] Repository clone ki ja rahi hai...{RESET}")
repo_dir = "/content/bot_deploy"
if os.path.exists(repo_dir):
  subprocess.run(f"rm -rf {repo_dir}", shell=True)

code, _, err = run_cmd(
    f"git clone -b {Git_Branch.strip()} {formatted_repo_url} {repo_dir}"
)
if code != 0:
  print(
      f"{RED}[❌] Git Clone fail ho gaya! Branch name ya Personal Access Token check karein.{RESET}"
  )
  sys.exit(1)

os.chdir(repo_dir)

print(f"{YELLOW}[4/5] Heroku app link ki ja rahi hai...{RESET}")
run_cmd('git config --global user.email "deployer@colab.local"')
run_cmd('git config --global user.name "Colab Deployer"')

app_name = Heroku_App_Name.strip().lower()
create_code, out, err = run_cmd(f"heroku create {app_name}", check=False)
if create_code != 0:
  if "Name is already taken" in err or "Name is already taken" in out:
    print(
        f"{RED}[!] '{app_name}' naam pehle se kisi aur ka hai. Heroku_App_Name me kuch naya/unique naam dalein!{RESET}"
    )
    sys.exit(1)
  else:
    run_cmd(f"heroku git:remote -a {app_name}")

print(f"{GREEN}[5/5] Heroku par code push shuru ho raha hai...{RESET}\n")
deploy = subprocess.Popen(
    f"git push https://git.heroku.com/{app_name}.git HEAD:main --force",
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
for line in deploy.stdout:
  print(line, end="")
deploy.wait()

if deploy.returncode == 0:
  print(f"\n{GREEN}✅ Successfully Deployed!{RESET}")
  if Auto_Scale:
    run_cmd(f"heroku ps:scale {Dyno_Type}=1 -a {app_name}")
    print(f"{GREEN}Dyno Start ho gaya!{RESET}")
  if Stream_Logs:
    time.sleep(2)
    log_proc = subprocess.Popen(
        f"heroku logs --tail -a {app_name}",
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    try:
      for log_line in log_proc.stdout:
        print(log_line, end="")
    except KeyboardInterrupt:
      pass
else:
  print(f"\n{RED}❌ Build fail hui. Upar build logs check karein.{RESET}")

Streaming output truncated to the last 5000 lines.
2026-09-06T00:04:28.630211+00:00 app[web.1]: For further information visit https://errors.pydantic.dev/2.13/v/missing
2026-09-06T00:04:28.718062+00:00 app[web.1]: INFO:     Waiting for child process [269]
2026-09-06T00:04:28.718086+00:00 app[web.1]: INFO:     Child process [269] died
2026-09-06T00:04:28.718892+00:00 app[web.1]: INFO:     Waiting for child process [275]
2026-09-06T00:04:28.718921+00:00 app[web.1]: INFO:     Child process [275] died
2026-09-06T00:04:29.135167+00:00 app[web.1]: Process SpawnProcess-91:
2026-09-06T00:04:29.136572+00:00 app[web.1]: Traceback (most recent call last):
2026-09-06T00:04:29.136635+00:00 app[web.1]: File "/app/.heroku/python/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
2026-09-06T00:04:29.136636+00:00 app[web.1]: self.run()
2026-09-06T00:04:29.136636+00:00 app[web.1]: File "/app/.heroku/python/lib/python3.10/multiprocessing/process.py", line 108, in run
2026-09-06T00:04:29.